# KG Consensus Neo4j Ingest

Loads `final-consensus/kg/*.consensus.json` and ingests the consensus knowledge graphs into Neo4j.

Required environment variables:

- `NEO4J_URI`
- `NEO4J_USER`
- `NEO4J_PASS`

Optional:

- `NEO4J_DB`, default `neo4j`
- `CONSENSUS_DIR`, default `./final-consensus` or current dir if already inside `final-consensus`
- `CONSENSUS_KG_GLOB`, default `*.consensus.json`
- `NEO4J_CLEAR_CONSENSUS`, default `0`. Set to `1` to delete existing nodes for the grades being ingested before writing.

In [17]:
import json
import os
import re
from pathlib import Path

from neo4j import GraphDatabase

try:
    from dotenv import load_dotenv

    load_dotenv()
except ModuleNotFoundError:
    pass


# =========================
# 0) CONFIG
# =========================
PROJECT_DIR = Path(os.getenv("PROJECT_DIR", Path.cwd())).expanduser()
CONSENSUS_DIR = Path(os.getenv("CONSENSUS_DIR", PROJECT_DIR / "final-consensus")).expanduser()
if not CONSENSUS_DIR.exists() and PROJECT_DIR.name == "final-consensus":
    CONSENSUS_DIR = PROJECT_DIR

KG_DIR = CONSENSUS_DIR / "kg"
CONSENSUS_KG_GLOB = os.getenv("CONSENSUS_KG_GLOB", "*.consensus.json")
CLEAR_EXISTING = os.getenv("NEO4J_CLEAR_CONSENSUS", "1").strip().lower() in {"1", "true", "yes"}

NEO4J_URI = os.getenv("NEO4J_URI", "").strip()
NEO4J_USER = os.getenv("NEO4J_USER", "").strip()
NEO4J_PASS = os.getenv("NEO4J_PASS", "").strip()
NEO4J_DB = os.getenv("NEO4J_DB", "772674a6").strip()


def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(CONSENSUS_DIR))
    except ValueError:
        return path.name


missing_env = [name for name, value in {
    "NEO4J_URI": NEO4J_URI,
    "NEO4J_USER": NEO4J_USER,
    "NEO4J_PASS": NEO4J_PASS,
}.items() if not value]
if missing_env:
    raise EnvironmentError("Missing required Neo4j environment variable(s): " + ", ".join(missing_env))

CONSENSUS_FILES = sorted(KG_DIR.glob(CONSENSUS_KG_GLOB))
if not CONSENSUS_FILES:
    raise FileNotFoundError(f"No consensus KG files found in {display_path(KG_DIR)} with pattern {CONSENSUS_KG_GLOB!r}")

print("Consensus files:")
for path in CONSENSUS_FILES:
    print("-", display_path(path))
print("Neo4j URI:", NEO4J_URI)
print("Neo4j database:", NEO4J_DB)
print("Clear existing course graph first:", CLEAR_EXISTING)

Consensus files:
- kg/Biologi Kelas XII.consensus.json
- kg/Fisika Kelas XII.consensus.json
- kg/Kimia Kelas XII.consensus.json
Neo4j URI: neo4j+ssc://772674a6.databases.neo4j.io
Neo4j database: 772674a6
Clear existing course graph first: True


In [18]:
# =========================
# 1) LOAD DATA
# =========================
def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


data_to_ingest = {}
for path in CONSENSUS_FILES:
    data = load_json(path)
    grade = data.get("grade") or path.name.removesuffix(".consensus.json")
    data["grade"] = grade
    data_to_ingest[grade] = data

print("Loaded consensus graphs:")
for grade, graph in data_to_ingest.items():
    print(f"- {grade}: {len(graph.get('chapters', []))} chapters")

Loaded consensus graphs:
- Biologi Kelas XII: 4 chapters
- Fisika Kelas XII: 9 chapters
- Kimia Kelas XII: 4 chapters


In [19]:
# =========================
# 2) HELPERS
# =========================
def safe_rel(rel_type: str) -> str:
    text = str(rel_type or "RELATED_TO").upper().replace(" ", "_")
    text = re.sub(r"[^A-Z0-9_]", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    if not text:
        return "RELATED_TO"
    if text[0].isdigit():
        text = "REL_" + text
    return text


def compact_json(value) -> str:
    if value in (None, ""):
        return ""
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))


def relation_props(rel: dict, *, grade: str, chapter: str, source: str, target: str) -> dict:
    review = rel.get("expert_review") or {}
    added_by = rel.get("added_by_reviewers") or review.get("added_by_reviewers") or []
    return {
        "relation_key": "|".join([grade, chapter, source, str(rel.get("type", "")), target]),
        "relation_type": str(rel.get("type", "")),
        "description": str(rel.get("description", "") or ""),
        "provenance": str(rel.get("provenance", "") or ""),
        "source": "consensus",
        "graph_role": "semantic",
        "consensus": str(review.get("consensus", "") or ""),
        "status": str(review.get("status", "") or ""),
        "n_reviewers": int(review.get("n_reviewers", 0) or 0),
        "ratings_json": compact_json(review.get("ratings", {})),
        "comments_json": compact_json(review.get("comments", {})),
        "expert_review_json": compact_json(review),
        "added_by_reviewers_json": compact_json(added_by),
    }


def collect_concept_names(graph: dict) -> set[str]:
    names = set()
    for chapter in graph.get("chapters", []):
        for subtopic in chapter.get("subtopics", []):
            for concept in subtopic.get("concepts", []):
                name = str(concept.get("name", "") or "").strip()
                if name:
                    names.add(name)
    return names


concept_names_by_grade = {grade: collect_concept_names(graph) for grade, graph in data_to_ingest.items()}
for grade, names in concept_names_by_grade.items():
    print(f"{grade}: {len(names)} concepts")

Biologi Kelas XII: 81 concepts
Fisika Kelas XII: 133 concepts
Kimia Kelas XII: 129 concepts


In [20]:
# =========================
# 3) CONNECT + CONSTRAINTS
# =========================
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
driver.verify_connectivity()
print("Connected to Neo4j")

constraint_queries = [
    "CREATE CONSTRAINT consensus_grade_name IF NOT EXISTS FOR (g:Grade) REQUIRE g.name IS UNIQUE",
    "CREATE CONSTRAINT consensus_chapter_key IF NOT EXISTS FOR (c:Chapter) REQUIRE (c.grade, c.name) IS UNIQUE",
    "CREATE CONSTRAINT consensus_subtopic_key IF NOT EXISTS FOR (s:Subtopic) REQUIRE (s.grade, s.chapter, s.name) IS UNIQUE",
    "CREATE CONSTRAINT consensus_concept_key IF NOT EXISTS FOR (c:Concept) REQUIRE (c.grade, c.name) IS UNIQUE",
    "CREATE CONSTRAINT consensus_concept_target_key IF NOT EXISTS FOR (t:ConceptTarget) REQUIRE (t.grade, t.name) IS UNIQUE",
]

with driver.session(database=NEO4J_DB) as session:
    for query in constraint_queries:
        session.run(query)

print("Constraints ensured")

Connected to Neo4j
Constraints ensured


In [21]:
# =========================
# 4) OPTIONAL CLEAR
# =========================
if CLEAR_EXISTING:
    print("Clearing existing graph for selected grades...")
    with driver.session(database=NEO4J_DB) as session:
        for grade in data_to_ingest:
            session.run("MATCH (n {grade: $grade}) DETACH DELETE n", grade=grade)
            session.run("MATCH (g:Grade {name: $grade}) DETACH DELETE g", grade=grade)
else:
    print("Skipping clear step; ingest will MERGE/SET existing data.")

Clearing existing graph for selected grades...


In [22]:
# =========================
# 5) PASS 1: NODES
# =========================
node_counts = {"Grade": 0, "Chapter": 0, "Subtopic": 0, "Concept": 0}

with driver.session(database=NEO4J_DB) as session:
    for grade, graph in data_to_ingest.items():
        session.run(
            "MERGE (g:Grade {name: $name}) SET g.type = $type, g.source = 'consensus', g.graph_role = 'structure'",
            name=grade,
            type="KLS_XII",
        )
        node_counts["Grade"] += 1

        for chapter in graph.get("chapters", []):
            chapter_name = str(chapter.get("chapter", "") or "").strip()
            if not chapter_name:
                continue
            session.run(
                """
                MERGE (c:Chapter {name: $name, grade: $grade})
                SET c.summary = $summary,
                    c.previous = $previous,
                    c.next = $next,
                    c.subchapters_json = $subchapters_json,
                    c.source = 'consensus',
                    c.graph_role = 'structure'
                WITH c
                MATCH (g:Grade {name: $grade})
                MERGE (g)-[r:HAS_CHAPTER]->(c)
                SET r.source = 'consensus', r.graph_role = 'structure'
                """,
                name=chapter_name,
                grade=grade,
                summary=chapter.get("chapter_summary", ""),
                previous=chapter.get("previous"),
                next=chapter.get("next"),
                subchapters_json=compact_json(chapter.get("subchapters", [])),
            )
            node_counts["Chapter"] += 1

            if chapter.get("next"):
                session.run(
                    """
                    MATCH (a:Chapter {name: $source, grade: $grade})
                    MATCH (b:Chapter {name: $target, grade: $grade})
                    MERGE (a)-[r:NEXT_CHAPTER]->(b)
                    SET r.source = 'consensus', r.graph_role = 'structure'
                    """,
                    source=chapter_name,
                    target=chapter["next"],
                    grade=grade,
                )

            for subtopic in chapter.get("subtopics", []):
                subtopic_name = str(subtopic.get("name", "") or "").strip()
                if not subtopic_name:
                    continue
                session.run(
                    """
                    MERGE (s:Subtopic {name: $name, chapter: $chapter, grade: $grade})
                    SET s.source = 'consensus',
                        s.graph_role = 'structure'
                    WITH s
                    MATCH (c:Chapter {name: $chapter, grade: $grade})
                    MERGE (c)-[r:HAS_SUBTOPIC]->(s)
                    SET r.source = 'consensus', r.graph_role = 'structure'
                    """,
                    name=subtopic_name,
                    chapter=chapter_name,
                    grade=grade,
                )
                node_counts["Subtopic"] += 1

                for concept in subtopic.get("concepts", []):
                    concept_name = str(concept.get("name", "") or "").strip()
                    if not concept_name:
                        continue
                    session.run(
                        """
                        MERGE (k:Concept {name: $name, grade: $grade})
                        SET k.description = $description,
                            k.glossary_validated = $glossary_validated,
                            k.materi_pokok_ref = $materi_pokok_ref,
                            k.provenance = $provenance,
                            k.source = 'consensus',
                            k.graph_role = 'concept'
                        WITH k
                        MATCH (s:Subtopic {name: $subtopic, chapter: $chapter, grade: $grade})
                        MERGE (s)-[r:HAS_CONCEPT]->(k)
                        SET r.source = 'consensus', r.graph_role = 'structure'
                        """,
                        name=concept_name,
                        grade=grade,
                        description=concept.get("description", ""),
                        glossary_validated=bool(concept.get("glossary_validated", False)),
                        materi_pokok_ref=str(concept.get("materi_pokok_ref", "") or ""),
                        provenance=str(concept.get("provenance", "") or ""),
                        subtopic=subtopic_name,
                        chapter=chapter_name,
                    )
                    node_counts["Concept"] += 1

print("Node pass complete:", node_counts)

Node pass complete: {'Grade': 3, 'Chapter': 17, 'Subtopic': 87, 'Concept': 347}


In [23]:
# =========================
# 6) PASS 2: INTRA-BOOK RELATIONS
# =========================
relation_counts = {"concept_to_concept": 0, "concept_to_target": 0, "chapter_relations": 0, "skipped": 0}

with driver.session(database=NEO4J_DB) as session:
    for grade, graph in data_to_ingest.items():
        known_concepts = concept_names_by_grade[grade]
        for chapter in graph.get("chapters", []):
            chapter_name = str(chapter.get("chapter", "") or "").strip()
            for subtopic in chapter.get("subtopics", []):
                for concept in subtopic.get("concepts", []):
                    source_name = str(concept.get("name", "") or "").strip()
                    if not source_name:
                        continue
                    for rel in concept.get("relations", []) or []:
                        rel_type = str(rel.get("type", "") or "").strip()
                        target_name = str(rel.get("target", rel.get("target_concept", "")) or "").strip()
                        if not rel_type or not target_name:
                            relation_counts["skipped"] += 1
                            continue
                        rel_label = safe_rel(rel_type)
                        props = relation_props(rel, grade=grade, chapter=chapter_name, source=source_name, target=target_name)
                        if target_name in known_concepts:
                            session.run(
                                f"""
                                MATCH (src:Concept {{name: $source, grade: $grade}})
                                MATCH (tgt:Concept {{name: $target, grade: $grade}})
                                MERGE (src)-[r:{rel_label} {{relation_key: $relation_key}}]->(tgt)
                                SET r += $props
                                """,
                                source=source_name,
                                target=target_name,
                                grade=grade,
                                relation_key=props["relation_key"],
                                props=props,
                            )
                            relation_counts["concept_to_concept"] += 1
                        else:
                            session.run(
                                "MERGE (t:ConceptTarget {name: $name, grade: $grade}) SET t.source = 'consensus', t.graph_role = 'target'",
                                name=target_name,
                                grade=grade,
                            )
                            session.run(
                                f"""
                                MATCH (src:Concept {{name: $source, grade: $grade}})
                                MATCH (tgt:ConceptTarget {{name: $target, grade: $grade}})
                                MERGE (src)-[r:{rel_label} {{relation_key: $relation_key}}]->(tgt)
                                SET r += $props
                                """,
                                source=source_name,
                                target=target_name,
                                grade=grade,
                                relation_key=props["relation_key"],
                                props=props,
                            )
                            relation_counts["concept_to_target"] += 1

            for chapter_rel in chapter.get("chapter_relations", []) or []:
                rel_type = str(chapter_rel.get("type", "") or "").strip()
                target_chapter = str(chapter_rel.get("target_chapter", "") or "").strip()
                if not rel_type or not target_chapter:
                    continue
                rel_label = safe_rel(rel_type)
                relation_key = "|".join([grade, chapter_name, rel_type, target_chapter])
                session.run(
                    f"""
                    MATCH (src:Chapter {{name: $source, grade: $grade}})
                    MATCH (tgt:Chapter {{name: $target, grade: $grade}})
                    MERGE (src)-[r:{rel_label} {{relation_key: $relation_key}}]->(tgt)
                    SET r.description = $description,
                        r.relation_type = $relation_type,
                        r.concept_links_json = $concept_links_json,
                        r.source = 'consensus',
                        r.graph_role = 'structure'
                    """,
                    source=chapter_name,
                    target=target_chapter,
                    grade=grade,
                    relation_key=relation_key,
                    description=chapter_rel.get("description", ""),
                    relation_type=rel_type,
                    concept_links_json=compact_json(chapter_rel.get("concept_links", [])),
                )
                relation_counts["chapter_relations"] += 1

print("Intra-book relation pass complete:", relation_counts)

Intra-book relation pass complete: {'concept_to_concept': 128, 'concept_to_target': 472, 'chapter_relations': 22, 'skipped': 0}


In [24]:
# =========================
# 7) PASS 3: CROSS-BOOK LINKS
# =========================
cross_counts = {"created": 0, "skipped_same_grade": 0, "skipped_unknown_grade": 0, "skipped_incomplete": 0}

with driver.session(database=NEO4J_DB) as session:
    for grade, graph in data_to_ingest.items():
        for chapter in graph.get("chapters", []):
            chapter_name = str(chapter.get("chapter", "") or "").strip()
            for subtopic in chapter.get("subtopics", []):
                for concept in subtopic.get("concepts", []):
                    source_name = str(concept.get("name", "") or "").strip()
                    if not source_name:
                        continue
                    for link in concept.get("cross_book_links", []) or []:
                        target_grade = str(link.get("target_book", "") or "").strip()
                        target_concept = str(link.get("target_concept", "") or "").strip()
                        if not target_grade or not target_concept:
                            cross_counts["skipped_incomplete"] += 1
                            continue
                        if target_grade == grade:
                            cross_counts["skipped_same_grade"] += 1
                            continue
                        if target_grade not in data_to_ingest:
                            cross_counts["skipped_unknown_grade"] += 1
                            continue
                        raw_type = str(link.get("relation_type", "LINTAS_BUKU") or "LINTAS_BUKU").strip()
                        rel_type = raw_type if raw_type.startswith("LINTAS_BUKU") else f"LINTAS_BUKU_{raw_type}"
                        rel_label = safe_rel(rel_type)
                        relation_key = "|".join([grade, source_name, rel_type, target_grade, target_concept])
                        session.run(
                            "MERGE (tgt:Concept {name: $name, grade: $grade}) SET tgt.source = coalesce(tgt.source, 'consensus'), tgt.graph_role = coalesce(tgt.graph_role, 'concept')",
                            name=target_concept,
                            grade=target_grade,
                        )
                        session.run(
                            f"""
                            MATCH (src:Concept {{name: $source, grade: $source_grade}})
                            MATCH (tgt:Concept {{name: $target, grade: $target_grade}})
                            MERGE (src)-[r:{rel_label} {{relation_key: $relation_key}}]->(tgt)
                            SET r.description = $description,
                                r.relation_type = $relation_type,
                                r.source = 'consensus',
                                r.graph_role = 'semantic'
                            """,
                            source=source_name,
                            source_grade=grade,
                            target=target_concept,
                            target_grade=target_grade,
                            relation_key=relation_key,
                            description=link.get("explanation", ""),
                            relation_type=rel_type,
                        )
                        cross_counts["created"] += 1

print("Cross-book pass complete:", cross_counts)

Cross-book pass complete: {'created': 0, 'skipped_same_grade': 0, 'skipped_unknown_grade': 0, 'skipped_incomplete': 0}


## Modularity / Dashboard Notes

For graph analytics, project the semantic KG separately from the storage structure. The ingest stores `Grade`, `Chapter`, and `Subtopic` nodes so browsing is easy, but those structure edges should not be mixed into modularity/community detection. Use `graph_role = "semantic"` and usually only `(:Concept)-[:...]->(:Concept)` relationships for modularity. `ConceptTarget` nodes are useful for preserving literal or not-yet-normalized targets, but they behave like many leaf nodes and can distort community metrics.


In [25]:
# =========================
# 8) VERIFY + CLOSE
# =========================
print("Verification:")
with driver.session(database=NEO4J_DB) as session:
    for label in ["Grade", "Chapter", "Subtopic", "Concept", "ConceptTarget"]:
        count = session.run(f"MATCH (n:{label}) RETURN count(n) AS count").single()["count"]
        print(f"- {label}: {count}")
    total_rels = session.run("MATCH ()-[r]->() RETURN count(r) AS count").single()["count"]
    consensus_rels = session.run("MATCH ()-[r]->() WHERE r.source = 'consensus' OR r.consensus IS NOT NULL RETURN count(r) AS count").single()["count"]
    expert_added = session.run("MATCH ()-[r]->() WHERE r.provenance = 'expert-added' RETURN count(r) AS count").single()["count"]
    print(f"- Total relationships: {total_rels}")
    print(f"- Consensus-tagged relationships: {consensus_rels}")
    print(f"- Expert-added relationships: {expert_added}")

driver.close()
print("Neo4j consensus ingest complete.")

Verification:
- Grade: 3
- Chapter: 17
- Subtopic: 87
- Concept: 343
- ConceptTarget: 426
- Total relationships: 1058
- Consensus-tagged relationships: 1058
- Expert-added relationships: 25
- Semantic Concept -> Concept relationships: 128
- Node graph roles: [{'graph_role': 'concept', 'count': 343}, {'graph_role': 'structure', 'count': 107}, {'graph_role': 'target', 'count': 426}]
- Relationship graph roles: [{'graph_role': 'semantic', 'count': 585}, {'graph_role': 'structure', 'count': 473}]

Recommended modularity projection:
Use only semantic Concept -> Concept relationships for community/modularity metrics.
Do not include Grade/Chapter/Subtopic structure edges or ConceptTarget leaves unless you intentionally want to measure the whole storage graph.

Cypher edge filter:
MATCH (:Concept)-[r]->(:Concept)
WHERE r.graph_role = 'semantic'
RETURN count(r)

Neo4j consensus ingest complete.
